In [ ]:
### Step 1 — Setup
import sys, os
sys.path.insert(0, os.path.expanduser("~/Sleep_Stage_Research/conference"))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader

import config as cfg
from data_utils import compute_class_weights, compute_channel_stats, load_subjects_from_h5, load_participant_info
from dataset import SleepSequenceDataset
from model import SleepStageNet, count_parameters
from trainer import Trainer
from eval_utils import (predict_on_subjects, compute_fold_metrics, aggregate_cv_results,
                        print_metrics, print_cv_summary, plot_confusion_matrix,
                        plot_training_history, plot_per_subject_kappa, save_results)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

h5_path = os.path.join(cfg.PREPROCESSED_DIR, "dreamt_psg7ch_epochs.h5")
folds_path = os.path.join(cfg.CHECKPOINT_DIR, "fold_assignments.json")

assert os.path.exists(h5_path), f"Run exp0_preprocess first! Missing: {h5_path}"
assert os.path.exists(folds_path), f"Run exp0_preprocess first! Missing: {folds_path}"

with open(folds_path) as f:
    folds = json.load(f)
print(f"Loaded {len(folds)} fold assignments")

EXP_NAME = "exp0_1"
RNN_MODE = "lstm_bilstm"

model_tmp = SleepStageNet(n_channels=len(cfg.PSG_CHANNELS), rnn_mode=RNN_MODE)
total_p, train_p = count_parameters(model_tmp)
print(f"\nModel: SleepStageNet (rnn_mode={RNN_MODE})")
print(f"  Total parameters: {total_p:,}")
print(f"  Trainable parameters: {train_p:,}")
print(model_tmp)
del model_tmp

In [ ]:
### Step 2 — Train all 5 folds
all_fold_results = []
all_histories = []

for fold_idx in range(cfg.NUM_FOLDS):
    print(f"\n{'#'*70}")
    print(f"# {EXP_NAME} | FOLD {fold_idx+1}/{cfg.NUM_FOLDS} | rnn_mode={RNN_MODE}")
    print(f"{'#'*70}")

    train_subjects = folds[str(fold_idx)]["train"]
    test_subjects = folds[str(fold_idx)]["test"]

    np.random.seed(cfg.SEED + fold_idx)
    n_val = max(1, int(len(train_subjects) * cfg.VAL_RATIO))
    perm = np.random.permutation(len(train_subjects))
    val_subjects = [train_subjects[i] for i in perm[:n_val]]
    actual_train = [train_subjects[i] for i in perm[n_val:]]

    print(f"  Train: {len(actual_train)} | Val: {len(val_subjects)} | Test: {len(test_subjects)}")

    # Reuse channel stats from exp0 (same data split)
    stats_path = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "channel_stats.npz")
    if os.path.exists(stats_path):
        stats = np.load(stats_path)
        mean, std = stats["mean"], stats["std"]
        print(f"  [LOAD] Channel stats from exp0_fold{fold_idx}")
    else:
        print(f"  Computing channel stats...")
        mean, std = compute_channel_stats(h5_path, actual_train)

    _, train_labels, _ = load_subjects_from_h5(h5_path, actual_train)
    class_weights = compute_class_weights(train_labels)
    del train_labels

    train_ds = SleepSequenceDataset(h5_path, actual_train, mean=mean, std=std)
    val_ds = SleepSequenceDataset(h5_path, val_subjects, mean=mean, std=std)
    print(f"  Train sequences: {len(train_ds)}, Val sequences: {len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=True)

    model = SleepStageNet(n_channels=len(cfg.PSG_CHANNELS), rnn_mode=RNN_MODE).to(device)
    trainer = Trainer(model, train_loader, val_loader, class_weights,
                      exp_name=EXP_NAME, fold=fold_idx, device=device)

    history = trainer.train()
    all_histories.append(history)

    print(f"\n  [EVAL] Evaluating on {len(test_subjects)} test subjects...")
    trainer.load_best_model()
    results = predict_on_subjects(model, h5_path, test_subjects, mean, std, device=device)
    overall, per_subj = compute_fold_metrics(results)
    print_metrics(overall, title=f"{EXP_NAME} Fold {fold_idx} Test Results")
    all_fold_results.append((overall, per_subj))

    fold_results_path = os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_fold{fold_idx}", "test_results.json")
    save_results({"overall": overall, "per_subject": {s: m for s, m in per_subj.items()}}, fold_results_path)

    del model, trainer, train_ds, val_ds, train_loader, val_loader
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*70}")
print(f"{EXP_NAME} ALL FOLDS COMPLETE")
print(f"{'='*70}")


In [ ]:
### Step 3 — CV summary

summary = aggregate_cv_results(all_fold_results)
print_cv_summary(summary)
save_results(summary, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_cv_summary.json"))

In [ ]:
### Step 4 — Visualizations
import matplotlib
matplotlib.rcParams.update({"font.size": 11})

for fold_idx, history in enumerate(all_histories):
    if history and history.get("train_loss"):
        plot_training_history(
            history, title=f"{EXP_NAME} Fold {fold_idx}",
            save_path=os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_fold{fold_idx}", "training_curves.png"))

cm_total = np.zeros((cfg.NUM_CLASSES, cfg.NUM_CLASSES), dtype=np.int64)
all_per_subj = {}
for overall, per_subj in all_fold_results:
    cm_total += np.array(overall["confusion_matrix"])
    all_per_subj.update(per_subj)

plot_confusion_matrix(cm_total, title=f"{EXP_NAME}: CNN+LSTM+BiLSTM - Confusion Matrix",
                      save_path=os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_confusion_matrix.png"))

plot_per_subject_kappa(all_per_subj, title=f"{EXP_NAME}: Per-Subject Kappa",
                       save_path=os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_per_subject_kappa.png"))

In [ ]:
### Step 5 — Experiment 1

pinfo = load_participant_info()
demo_map = {}
for _, row in pinfo.iterrows():
    demo_map[row["SID"]] = {"gender": row["GENDER"], "age": row["AGE"], "bmi": row["BMI"], "ahi": row["AHI"]}

def get_subgroup(sid, axis):
    d = demo_map.get(sid)
    if d is None:
        return None
    if axis == "gender":
        return d["gender"]
    elif axis == "age":
        return "<50" if d["age"] < 50 else "50-65" if d["age"] <= 65 else ">65"
    elif axis == "ahi":
        return "AHI<15" if d["ahi"] < 15 else "AHI>=15"
    return None

for axis in ["gender", "age", "ahi"]:
    print(f"\n{'='*60}")
    print(f"  {EXP_NAME} STRATIFIED ANALYSIS: {axis.upper()}")
    print(f"{'='*60}")

    group_metrics = {}
    for grp in sorted(set(get_subgroup(s, axis) for s in all_per_subj if get_subgroup(s, axis))):
        members = [sid for sid in all_per_subj if get_subgroup(sid, axis) == grp]
        kappas = [all_per_subj[sid]["kappa"] for sid in members]
        accs = [all_per_subj[sid]["accuracy"] for sid in members]
        f1s = [all_per_subj[sid]["f1_macro"] for sid in members]

        print(f"\n  {grp} (n={len(members)}):")
        print(f"    Accuracy:   {np.mean(accs):.4f} \u00B1 {np.std(accs):.4f}")
        print(f"    F1 (macro): {np.mean(f1s):.4f} \u00B1 {np.std(f1s):.4f}")
        print(f"    Kappa:      {np.mean(kappas):.4f} \u00B1 {np.std(kappas):.4f}")

        group_metrics[grp] = {
            "n_subjects": len(members),
            "accuracy": {"mean": float(np.mean(accs)), "std": float(np.std(accs))},
            "f1_macro": {"mean": float(np.mean(f1s)), "std": float(np.std(f1s))},
            "kappa": {"mean": float(np.mean(kappas)), "std": float(np.std(kappas))},
        }

    save_results(group_metrics, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_stratified_{axis}.json"))

print(f"\n{EXP_NAME} + Stratified Analysis complete.")